# 🤖 Automated Candidate Interview & Evaluation System
## Colab Notebook — Groq + LLaMA + ChromaDB + LangChain ≥ 1.2

### 📋 Required API Keys (save in Colab Secrets ▶ 🔑)
| Secret Name | Where to get it | Used for |
|---|---|---|
| `GROQ_API_KEY` | https://console.groq.com | LLM calls (LLaMA Instant / Versatile) |
| `MEM0_API_KEY` | https://app.mem0.ai (optional) | Memory observability |

> **Models used:** `llama-3.1-8b-instant` (fast, low-token) · `llama-3.3-70b-versatile` (evaluation)
> **VectorDB:** ChromaDB (free, local, no API key needed)
> **LangChain:** ≥ 0.1.20 (modern `langchain-core` / `langchain-groq` imports)


## 📦 Phase 0 — Install Dependencies

In [15]:
# Install all required packages
!pip install -q \
    groq \
    langchain\
    langchain-core\
    langchain-groq \
    langchain-community \
    langchain-chroma\
    chromadb \
    'pydantic[email]' \
    mem0ai \
    sentence-transformers \
    python-dotenv \
    nest_asyncio
print('✅ All packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 5.0 MB/s eta 0:00:00
✅ All packages installed


## ⚙️ Phase 1 — Setup: API Keys & Project Structure

In [33]:
# ═══════════════════════════════════════════════════════════════
# LOAD GROQ API KEY FROM COLAB SECRETS
# ═══════════════════════════════════════════════════════════════

import os

try:
    # For Google Colab Secrets
    from google.colab import userdata

    GROQ_API_KEY = userdata.get('GROQ_API_KEY')

    if not GROQ_API_KEY:
        raise ValueError("❌ GROQ_API_KEY not found in Colab Secrets.")

    os.environ['GROQ_API_KEY'] = GROQ_API_KEY

    print("✅ GROQ_API_KEY loaded successfully from Colab Secrets.")

except Exception as e:
    print(f"❌ Error loading GROQ_API_KEY: {e}")

✅ GROQ_API_KEY loaded successfully from Colab Secrets.


## 💾 Phase 2 — Save All `.py` Files to Project Folders
Writes every module in the **exact same structure** as the original project.


In [8]:
# ═══════════════════════════════════════════════════════════════
# utilities/pydantic_models.py
# ═══════════════════════════════════════════════════════════════
pydantic_models_py = '''
from pydantic import BaseModel, EmailStr, Field, field_validator
from datetime import datetime
from typing import Optional, List

class SearchRequest(BaseModel):
    user_id: str = Field(..., min_length=3, max_length=50)
    email: EmailStr
    query: str = Field(..., min_length=1, max_length=200)
    tags: Optional[List[str]] = Field(default_factory=list)

    @field_validator("query")
    def query_must_not_be_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("Query must not be empty or whitespace")
        return value.strip()


class SearchResponse(BaseModel):
    status: str
    message: str
    result_count: int = Field(0, ge=0)
    results: List[dict] = Field(default_factory=list)
    processed_at: datetime = Field(default_factory=datetime.utcnow)


def build_search_response(request: SearchRequest) -> SearchResponse:
    example_results = [{"id": 1, "title": "Example item", "query": request.query}]
    return SearchResponse(
        status="success",
        message=f"Search completed for user {request.user_id}",
        result_count=len(example_results),
        results=example_results,
    )


def demo() -> None:
    payload = {
        "user_id": "user123",
        "email": "user@example.com",
        "query": "search for utilities",
        "tags": ["example", "demo"],
    }
    req = SearchRequest(**payload)
    resp = build_search_response(req)
    print("Request:", req.model_dump_json(indent=2))
    print("Response:", resp.model_dump_json(indent=2))
'''

(BASE_DIR / 'utilities' / 'pydantic_models.py').write_text(pydantic_models_py.strip())
print('✅ utilities/pydantic_models.py')


✅ utilities/pydantic_models.py


In [25]:
qvt_py = '''
import re
from typing import Dict

ALLOWED_QUERY_PATTERN = re.compile(r"^[a-zA-Z0-9\\s?@#\\-_.,'\\"()]+$")
STOP_WORDS = {"the", "is", "and", "or", "for", "a", "an", "to"}
SYNONYMS = {"buy": "purchase", "find": "search", "latest": "recent"}


def validate_query(query: str) -> bool:
    if not query or len(query.strip()) < 3:
        raise ValueError("Query must be at least 3 characters long.")
    if not ALLOWED_QUERY_PATTERN.match(query):
        raise ValueError("Query contains invalid characters.")
    return True


def transform_query(query: str) -> Dict[str, str]:
    normalized = re.sub(r"\\s+", " ", query.strip().lower())
    tokens = [SYNONYMS.get(t, t) for t in normalized.split() if t not in STOP_WORDS]
    cleaned = " ".join(tokens)
    return {
        "original": query,
        "normalized": normalized,
        "cleaned": cleaned,
        "signature": cleaned.replace(" ", "_"),
    }


def handle_query(query: str) -> Dict[str, str]:
    validate_query(query)
    return transform_query(query)
'''

(BASE_DIR / 'utilities' / 'query_validation_transformation.py').write_text(qvt_py.strip())
print('✅ utilities/query_validation_transformation.py')

✅ utilities/query_validation_transformation.py


In [19]:
# ═══════════════════════════════════════════════════════════════
# utilities/logging_example.py
# ═══════════════════════════════════════════════════════════════
logging_py = '''
import logging
import sys
from pathlib import Path


def get_app_logger(name: str = __name__, log_dir: str = "logs") -> logging.Logger:
    logger = logging.getLogger(name)
    if logger.handlers:
        return logger
    logger.setLevel(logging.DEBUG)

    console = logging.StreamHandler(sys.stdout)
    console.setLevel(logging.INFO)
    console.setFormatter(logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s"))

    Path(log_dir).mkdir(exist_ok=True)
    fh = logging.FileHandler(f"{log_dir}/utility_logging_example.log", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s"))

    logger.addHandler(console)
    logger.addHandler(fh)
    logger.propagate = False
    return logger


def run_logging_demo() -> None:
    logger = get_app_logger("utility_logger")
    logger.debug("Debugging values: %s", {"step": 1, "status": "starting"})
    logger.info("Application example started.")
    logger.warning("This is a warning example for the logging utility.")
    try:
        _ = 10 / 0
    except ZeroDivisionError:
        logger.exception("An exception occurred while dividing by zero.")
    logger.info("Logging demo finished.")
'''

(BASE_DIR / 'utilities' / 'logging_example.py').write_text(logging_py.strip())
print('✅ utilities/logging_example.py')


✅ utilities/logging_example.py


In [41]:
# ═══════════════════════════════════════════════════════════════
# utilities/mem0_example.py  (Groq + ChromaDB version)
# ═══════════════════════════════════════════════════════════════
mem0_py = '''
import os
from mem0 import Memory

if "SSL_CERT_FILE" in os.environ:
    del os.environ["SSL_CERT_FILE"]


def get_mem0_config(db_path: str = "db") -> dict:
    """Build mem0 config using Groq + ChromaDB (free, no OpenAI needed)."""
    return {
        "vector_store": {
            "provider": "chroma",
            "config": {"collection_name": "interview_memory", "path": db_path},
        },
        "llm": {
            "provider": "groq",
            "config": {
                "model": "llama-3.1-8b-instant",
                "temperature": 0,
                "max_tokens": 512,
            },
        },
        "embedder": {
            "provider": "huggingface",
            "config": {"model": "all-MiniLM-L6-v2"},
        },
    }


def run_observability_demo(db_path: str = "db") -> None:
    config = get_mem0_config(db_path)
    m = Memory.from_config(config)
    user_id = "candidate_123"

    print("\\n--- [Step 1] Storing Initial Preference ---")
    result = m.add("I prefer using FastAPI and AWS.", user_id=user_id)

    mem_id = None
    if isinstance(result, list) and result:
        mem_id = result[0].get("id")
    elif isinstance(result, dict):
        res_list = result.get("results") or result.get("memories") or []
        if res_list:
            mem_id = res_list[0].get("id")

    print("--- [Step 2] Updating Preference ---")
    m.add("Actually, I moved my projects to Google Cloud.", user_id=user_id)

    print("\\n--- [Step 3] Observability: Memory History ---")
    if mem_id:
        for entry in m.history(memory_id=mem_id):
            print(f"Event : {entry.get(\'event\')}")
            print(f"Old   : {entry.get(\'old_memory\') or \'Initial\'}")
            print(f"New   : {entry.get(\'memory\')}")
            print("-" * 30)
    else:
        print("Note: ID not captured. Check db/ folder.")

    print("\\n--- [Step 4] Final Memory Search ---")
    search_results = m.search(
        "What is my deployment preference?",
        filters={"user_id": user_id},
    )
    memories = (
        search_results.get("results")
        if isinstance(search_results, dict)
        else search_results
    )
    for r in (memories or []):
        val = r.get("memory") or r.get("payload", {}).get("value")
        print(f"Memory: {val}  (score: {r.get(\'score\')})")
'''

(BASE_DIR / 'utilities' / 'mem0_example.py').write_text(mem0_py.strip())
print('✅ utilities/mem0_example.py')

✅ utilities/mem0_example.py


In [38]:
# ═══════════════════════════════════════════════════════════════
# utilities/__init__.py
# ═══════════════════════════════════════════════════════════════
(BASE_DIR / 'utilities' / '__init__.py').write_text('')

# ═══════════════════════════════════════════════════════════════
# agent_core.py  — Groq-based agents replacing autogen/OpenAI
# ═══════════════════════════════════════════════════════════════
agent_core_py = '''
import os
from groq import Groq
from typing import List, Dict

client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

# Token budgets (Groq rate limits: 6000 TPM instant, 32768 context)
INSTANT_MODEL   = "llama-3.1-8b-instant"   # fast, cheap — interviewer / transformer
VERSATILE_MODEL = "llama-3.3-70b-versatile" # smarter — evaluator
MAX_TOKENS_FAST = 300
MAX_TOKENS_EVAL = 500


def chat(model: str, messages: List[Dict], max_tokens: int = MAX_TOKENS_FAST) -> str:
    """Single call wrapper with token guard."""
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=0.7,
    )
    return resp.choices[0].message.content.strip()


def interviewer_ask(job: str, history: List[Dict], q_num: int) -> str:
    system = (
        f"You are a professional interviewer for a {job} position. "
        f"Ask question {q_num} of 3 (technical/problem-solving/culture fit). "
        "Keep it under 50 words. If all 3 done, reply exactly: TERMINATE"
    )
    msgs = [{"role": "system", "content": system}] + history
    return chat(INSTANT_MODEL, msgs, MAX_TOKENS_FAST)


def evaluator_feedback(job: str, question: str, answer: str) -> str:
    system = (
        f"You are a career coach for {job} interviews. "
        "Give constructive feedback on the candidate answer in max 60 words."
    )
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": f"""Question: {question}
Answer: {answer}"""},
    ]
    return chat(VERSATILE_MODEL, msgs, MAX_TOKENS_EVAL)


def final_evaluator(job: str, transcript: List[Dict]) -> str:
    system = (
        f"You are a senior hiring manager for a {job} role. "
        "Summarise the candidate performance across all answers. "
        "Give: score /10, strengths, weaknesses, hire recommendation. Max 150 words."
    )
    summary_text = "\\n".join(
        [f"Q{i+1}: {t['question']}\\nA: {t['answer']}" for i, t in enumerate(transcript)]
    )
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": summary_text},
    ]
    return chat(VERSATILE_MODEL, msgs, MAX_TOKENS_EVAL)
'''

(BASE_DIR / 'agent_core.py').write_text(agent_core_py.strip())
print('✅ agent_core.py')

✅ agent_core.py


In [22]:
# ═══════════════════════════════════════════════════════════════
# main.py — orchestrator (Groq version)
# ═══════════════════════════════════════════════════════════════
main_py = '''
from utilities.pydantic_models import demo as pydantic_demo
from utilities.query_validation_transformation import handle_query
from utilities.logging_example import run_logging_demo
from utilities.mem0_example import run_observability_demo


def main():
    print("=" * 60)
    print("PHASE 1: Pydantic Models Demo")
    print("=" * 60)
    pydantic_demo()

    print("\n" + "=" * 60)
    print("PHASE 2: Query Validation & Transformation")
    print("=" * 60)
    result = handle_query("Find the latest AI trends and buy some books")
    for k, v in result.items():
        print(f"  {k:12}: {v}")

    print("\n" + "=" * 60)
    print("PHASE 3: Logging Demo")
    print("=" * 60)
    run_logging_demo()

    print("\n" + "=" * 60)
    print("PHASE 4: Mem0 Observability Demo")
    print("=" * 60)
    run_observability_demo()


if __name__ == "__main__":
    main()
'''

(BASE_DIR / 'main.py').write_text(main_py.strip())

# ── static files ──
static_script = open('/content/automated_interview_system/../' + '../dev/null', 'w') if False else None
print('✅ main.py')
print('\n📁 All .py files saved successfully!')
print('\nFile listing:')
for f in sorted(BASE_DIR.rglob('*.py')):
    print(f'  {f.relative_to(BASE_DIR)}')


✅ main.py

📁 All .py files saved successfully!

File listing:
  agent_core.py
  main.py
  utilities/__init__.py
  utilities/logging_example.py
  utilities/mem0_example.py
  utilities/pydantic_models.py
  utilities/query_validation_transformation.py


## 🧩 Phase 3 — Utility: Pydantic Request & Response Models
Validates input and structures output using Pydantic v2.


In [23]:
import sys
sys.path.insert(0, str(BASE_DIR))
os.chdir(BASE_DIR)

from utilities.pydantic_models import SearchRequest, SearchResponse, build_search_response, demo

print('=' * 60)
print('Pydantic Model Demo')
print('=' * 60)
demo()

# Phase output
PHASE3_OUTPUT = {
    'phase': 'Phase 3 — Pydantic Models',
    'status': 'completed',
    'request_fields': list(SearchRequest.model_fields.keys()),
    'response_fields': list(SearchResponse.model_fields.keys()),
}
phase3_path = BASE_DIR / 'outputs' / 'phase3_pydantic.json'
phase3_path.write_text(json.dumps(PHASE3_OUTPUT, indent=2))
print(f'\n✅ Phase 3 output saved → {phase3_path}')


Pydantic Model Demo
Request: {
  "user_id": "user123",
  "email": "user@example.com",
  "query": "search for utilities",
  "tags": [
    "example",
    "demo"
  ]
}
Response: {
  "status": "success",
  "message": "Search completed for user user123",
  "result_count": 1,
  "results": [
    {
      "id": 1,
      "title": "Example item",
      "query": "search for utilities"
    }
  ],
  "processed_at": "2026-05-07T03:44:49.672463"
}

✅ Phase 3 output saved → /content/automated_interview_system/outputs/phase3_pydantic.json


## 🔍 Phase 4 — Utility: Query Validation & Transformation
Validates queries, removes stop-words, applies synonym mapping.


In [26]:
from utilities.query_validation_transformation import handle_query, validate_query

test_queries = [
    'Find the latest AI trends and buy some books',
    'search for python developer jobs',
    'What are the best practices for FastAPI?',
]

phase4_results = []
print('=' * 60)
print('Query Validation & Transformation Demo')
print('=' * 60)
for q in test_queries:
    result = handle_query(q)
    phase4_results.append(result)
    print(f"\nInput    : {q}")
    for k, v in result.items():
        print(f"  {k:12}: {v}")

# Test invalid
try:
    validate_query('ab')
except ValueError as e:
    print(f"\n✅ Correctly rejected short query: {e}")

PHASE4_OUTPUT = {'phase': 'Phase 4 — Query Validation', 'status': 'completed', 'results': phase4_results}
phase4_path = BASE_DIR / 'outputs' / 'phase4_query.json'
phase4_path.write_text(json.dumps(PHASE4_OUTPUT, indent=2))
print(f'\n✅ Phase 4 output saved → {phase4_path}')


Query Validation & Transformation Demo

Input    : Find the latest AI trends and buy some books
  original    : Find the latest AI trends and buy some books
  normalized  : find the latest ai trends and buy some books
  cleaned     : search recent ai trends purchase some books
  signature   : search_recent_ai_trends_purchase_some_books

Input    : search for python developer jobs
  original    : search for python developer jobs
  normalized  : search for python developer jobs
  cleaned     : search python developer jobs
  signature   : search_python_developer_jobs

Input    : What are the best practices for FastAPI?
  original    : What are the best practices for FastAPI?
  normalized  : what are the best practices for fastapi?
  cleaned     : what are best practices fastapi?
  signature   : what_are_best_practices_fastapi?

✅ Correctly rejected short query: Query must be at least 3 characters long.

✅ Phase 4 output saved → /content/automated_interview_system/outputs/phase4_query.json

## 📝 Phase 5 — Utility: Logging Demo

In [27]:
from utilities.logging_example import run_logging_demo, get_app_logger

print('=' * 60)
print('Logging Demo')
print('=' * 60)
run_logging_demo()

log_file = BASE_DIR / 'logs' / 'utility_logging_example.log'
PHASE5_OUTPUT = {
    'phase': 'Phase 5 — Logging',
    'status': 'completed',
    'log_file': str(log_file),
}
phase5_path = BASE_DIR / 'outputs' / 'phase5_logging.json'
phase5_path.write_text(json.dumps(PHASE5_OUTPUT, indent=2))
print(f'\n✅ Phase 5 output saved → {phase5_path}')


Logging Demo
2026-05-07 03:46:19,569 - utility_logger - INFO - Application example started.
2026-05-07 03:46:19,570 - utility_logger - WARNING - This is a warning example for the logging utility.
2026-05-07 03:46:19,571 - utility_logger - ERROR - An exception occurred while dividing by zero.
Traceback (most recent call last):
  File "/content/automated_interview_system/utilities/logging_example.py", line 33, in run_logging_demo
    _ = 10 / 0
        ~~~^~~
ZeroDivisionError: division by zero
2026-05-07 03:46:19,573 - utility_logger - INFO - Logging demo finished.

✅ Phase 5 output saved → /content/automated_interview_system/outputs/phase5_logging.json


## 🗄️ Phase 6 — ChromaDB Vector Store (Free, Local)
Stores candidate profile embeddings for semantic search — no API key needed.


In [28]:
import chromadb
from chromadb.utils import embedding_functions

# Use free local embedding (no API key needed)
chroma_client = chromadb.PersistentClient(path=str(BASE_DIR / 'db'))

# Default embedding function (uses sentence-transformers locally)
ef = embedding_functions.DefaultEmbeddingFunction()

collection = chroma_client.get_or_create_collection(
    name='candidate_profiles',
    embedding_function=ef,
    metadata={'hnsw:space': 'cosine'},
)

# Add sample candidate profiles
profiles = [
    {'id': 'c1', 'text': 'Experienced Python developer with FastAPI and AWS skills', 'meta': {'role': 'backend', 'yoe': 5}},
    {'id': 'c2', 'text': 'ML engineer skilled in PyTorch, transformers, and data pipelines', 'meta': {'role': 'ml', 'yoe': 3}},
    {'id': 'c3', 'text': 'Full-stack developer with React, Node.js, and PostgreSQL', 'meta': {'role': 'fullstack', 'yoe': 4}},
    {'id': 'c4', 'text': 'DevOps engineer with Kubernetes, Docker, and CI/CD expertise', 'meta': {'role': 'devops', 'yoe': 6}},
]

existing = collection.get(ids=[p['id'] for p in profiles])['ids']
new_profiles = [p for p in profiles if p['id'] not in existing]
if new_profiles:
    collection.add(
        ids=[p['id'] for p in new_profiles],
        documents=[p['text'] for p in new_profiles],
        metadatas=[p['meta'] for p in new_profiles],
    )
    print(f'Added {len(new_profiles)} candidate profiles to ChromaDB')
else:
    print('Profiles already exist in ChromaDB')

# Semantic search
query = 'AI engineer with machine learning experience'
results = collection.query(query_texts=[query], n_results=2)
print(f'\nQuery: "{query}"')
print('Top matches:')
for doc, dist, meta in zip(results['documents'][0], results['distances'][0], results['metadatas'][0]):
    print(f'  [{meta["role"]}, {meta["yoe"]}y] {doc[:60]}... (dist={dist:.3f})')

PHASE6_OUTPUT = {
    'phase': 'Phase 6 — ChromaDB',
    'status': 'completed',
    'collection': 'candidate_profiles',
    'profiles_stored': collection.count(),
    'sample_query': query,
    'top_match': results['documents'][0][0] if results['documents'] else None,
}
phase6_path = BASE_DIR / 'outputs' / 'phase6_chromadb.json'
phase6_path.write_text(json.dumps(PHASE6_OUTPUT, indent=2))
print(f'\n✅ Phase 6 output saved → {phase6_path}')


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 52.9MiB/s]


Added 4 candidate profiles to ChromaDB

Query: "AI engineer with machine learning experience"
Top matches:
  [ml, 3y] ML engineer skilled in PyTorch, transformers, and data pipel... (dist=0.415)
  [backend, 5y] Experienced Python developer with FastAPI and AWS skills... (dist=0.672)

✅ Phase 6 output saved → /content/automated_interview_system/outputs/phase6_chromadb.json


## 🔗 Phase 7 — LangChain ≥ 1.2: Resume Analyzer Chain
Uses modern `langchain-core` / `langchain-groq` imports (v1.2+ style).


In [34]:
# LangChain >= 0.1.20 / 1.2 imports
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ── LLM — use Instant for speed/token saving ──
llm_fast = ChatGroq(
    model='llama-3.1-8b-instant',
    temperature=0.3,
    max_tokens=400,
    groq_api_key=GROQ_API_KEY,
)

# ── Resume Analysis Prompt (LangChain >= 1.2 style) ──
resume_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an expert HR analyst. Analyse the resume excerpt for a {job_role} position. '
     'Return JSON with keys: skills (list), experience_years (int), fit_score (1-10), summary (str). '
     'Max 200 tokens.'),
    ('human', 'Resume: {resume_text}'),
])

# ── Chain (LCEL pipe syntax) ──
resume_chain = resume_prompt | llm_fast | StrOutputParser()

# ── Test ──
sample_resume = (
    '5 years building REST APIs with FastAPI and Django. '
    'Deployed ML models on AWS SageMaker. '
    'Proficient in Python, Docker, PostgreSQL. '
    'Led a team of 3 engineers.'
)

print('Running resume analysis chain...')
analysis = resume_chain.invoke({'job_role': 'AI Engineer', 'resume_text': sample_resume})
print('\n📄 Resume Analysis Result:')
print(analysis)

PHASE7_OUTPUT = {
    'phase': 'Phase 7 — LangChain Resume Analyzer',
    'status': 'completed',
    'model': 'llama-3.1-8b-instant',
    'chain': 'ChatPromptTemplate | ChatGroq | StrOutputParser',
    'analysis_output': analysis,
}
phase7_path = BASE_DIR / 'outputs' / 'phase7_langchain.json'
phase7_path.write_text(json.dumps(PHASE7_OUTPUT, indent=2))
print(f'\n✅ Phase 7 output saved → {phase7_path}')


Running resume analysis chain...

📄 Resume Analysis Result:
```json
{
  "skills": [
    "FastAPI",
    "Django",
    "AWS SageMaker",
    "Python",
    "Docker",
    "PostgreSQL"
  ],
  "experience_years": 5,
  "fit_score": 8,
  "summary": "Highly skilled AI Engineer with 5 years of experience in building and deploying REST APIs, as well as leading a team of engineers. Proficient in a range of technologies including FastAPI, Django, AWS SageMaker, Python, Docker, and PostgreSQL."
}
```

The fit score of 8 is based on the following factors:

- Relevant experience: 3/5 (5 years of experience in AI-related roles)
- Technical skills: 4/5 (proficient in multiple relevant technologies)
- Leadership experience: 1/5 (led a team of 3 engineers, but no further details are provided)

✅ Phase 7 output saved → /content/automated_interview_system/outputs/phase7_langchain.json


## 🎙️ Phase 8 — Automated Interview Agent (Groq Multi-Agent)
Simulates a full 3-question interview with Interviewer → Candidate → Evaluator loop.
Uses `llama-3.1-8b-instant` for interviewer and `llama-3.3-70b-versatile` for evaluator.


In [39]:
from agent_core import interviewer_ask, evaluator_feedback, final_evaluator, INSTANT_MODEL, VERSATILE_MODEL

def run_automated_interview(job_position: str, candidate_answers: List[str]) -> Dict:
    """Run a simulated 3-question interview with pre-provided answers."""
    print(f'\n{'='*60}')
    print(f'🎙️  Interview: {job_position}')
    print(f'{'='*60}')

    history = []
    transcript = []

    for q_num in range(1, 4):
        # Interviewer asks
        question = interviewer_ask(job_position, history, q_num)
        print(f'\n[Q{q_num}] Interviewer: {question}')
        history.append({'role': 'assistant', 'content': question})

        if 'TERMINATE' in question.upper():
            break

        # Candidate answers (simulated)
        answer = candidate_answers[q_num - 1] if q_num - 1 < len(candidate_answers) else 'No answer provided.'
        print(f'   Candidate   : {answer}')
        history.append({'role': 'user', 'content': answer})

        # Evaluator gives feedback
        feedback = evaluator_feedback(job_position, question, answer)
        print(f'   Evaluator   : {feedback}')

        transcript.append({'q_num': q_num, 'question': question, 'answer': answer, 'feedback': feedback})

    # Final evaluation
    print(f'\n{"-"*60}')
    print('📊 FINAL EVALUATION')
    print(f'{"-"*60}')
    final_report = final_evaluator(job_position, transcript)
    print(final_report)

    return {'transcript': transcript, 'final_report': final_report}


# Simulated candidate answers
SIMULATED_ANSWERS = [
    'I have 4 years of experience building scalable Python APIs using FastAPI and deploying on AWS ECS. I am comfortable with async programming and have worked with PostgreSQL and Redis.',
    'I would start by profiling the bottleneck using cProfile, then check database queries for N+1 issues, add caching with Redis, and finally consider horizontal scaling with load balancers.',
    'I thrive in collaborative environments. I do daily standups, use async communication via Slack, and always document my work. I believe in code reviews as a learning tool.',
]

interview_result = run_automated_interview('AI Engineer', SIMULATED_ANSWERS)

PHASE8_OUTPUT = {
    'phase': 'Phase 8 — Interview Agent',
    'status': 'completed',
    'models_used': {'interviewer': INSTANT_MODEL, 'evaluator': VERSATILE_MODEL},
    'transcript': interview_result['transcript'],
    'final_report': interview_result['final_report'],
}
phase8_path = BASE_DIR / 'outputs' / 'phase8_interview.json'
phase8_path.write_text(json.dumps(PHASE8_OUTPUT, indent=2, default=str))
print(f'\n✅ Phase 8 output saved → {phase8_path}')



🎙️  Interview: AI Engineer

[Q1] Interviewer: Here's your first question:

You're tasked with building a chatbot that can understand and respond to user queries in multiple languages. Describe a simple language model you'd use and how you'd adapt it to support multiple languages.
   Candidate   : I have 4 years of experience building scalable Python APIs using FastAPI and deploying on AWS ECS. I am comfortable with async programming and have worked with PostgreSQL and Redis.
   Evaluator   : The answer is irrelevant to the question, lacking a language model description and multi-language adaptation approach. It focuses on unrelated skills and experience, failing to address the chatbot task.

[Q2] Interviewer: That's a good foundation. Here's your next question:

Suppose you're building a real-time analytics dashboard using WebSockets. Describe how you would handle a scenario where the WebSocket connection is interrupted, and you need to reconnect and synchronize the client's data in r

## 🧠 Phase 9 — Mem0 Memory Observability (Groq + ChromaDB)
Stores and tracks candidate preference evolution using free local stack.


In [42]:
from utilities.mem0_example import run_observability_demo

print('Running Mem0 observability demo...')
try:
    run_observability_demo(db_path=str(BASE_DIR / 'db'))
    mem0_status = 'completed'
except Exception as e:
    print(f'⚠️  Mem0 demo error: {e}')
    print('This may require MEM0_API_KEY for cloud features — local ChromaDB mode attempted.')
    mem0_status = f'error: {str(e)}'

PHASE9_OUTPUT = {'phase': 'Phase 9 — Mem0 Observability', 'status': mem0_status}
phase9_path = BASE_DIR / 'outputs' / 'phase9_mem0.json'
phase9_path.write_text(json.dumps(PHASE9_OUTPUT, indent=2))
print(f'\n✅ Phase 9 output saved → {phase9_path}')


Running Mem0 observability demo...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reus

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⚠️  Mem0 demo error: An instance of Chroma already exists for /content/automated_interview_system/db with different settings
This may require MEM0_API_KEY for cloud features — local ChromaDB mode attempted.

✅ Phase 9 output saved → /content/automated_interview_system/outputs/phase9_mem0.json


/usr/local/lib/python3.12/dist-packages/mem0/embeddings/huggingface.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.config.embedding_dims = self.config.embedding_dims or self.model.get_sentence_embedding_dimension()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 🔎 Phase 10 — LangChain RAG: Candidate Q&A over ChromaDB
Retrieval-Augmented Generation — asks questions over stored candidate profiles.


In [43]:
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Free local embeddings
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Load existing ChromaDB collection
vectorstore = Chroma(
    client=chroma_client,
    collection_name='candidate_profiles',
    embedding_function=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

llm_versatile = ChatGroq(
    model='llama-3.3-70b-versatile',
    temperature=0.3,
    max_tokens=300,
    groq_api_key=GROQ_API_KEY,
)

rag_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a hiring assistant. Use the candidate profiles below to answer the question. '
     'Context: {context}'),
    ('human', '{question}'),
])

def format_docs(docs):
    return '\n---\n'.join(d.page_content for d in docs)

rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt
    | llm_versatile
    | StrOutputParser()
)

rag_question = 'Which candidate is best suited for a machine learning role?'
print(f'Question: {rag_question}')
rag_answer = rag_chain.invoke(rag_question)
print(f'\nAnswer: {rag_answer}')

PHASE10_OUTPUT = {
    'phase': 'Phase 10 — LangChain RAG',
    'status': 'completed',
    'question': rag_question,
    'answer': rag_answer,
}
phase10_path = BASE_DIR / 'outputs' / 'phase10_rag.json'
phase10_path.write_text(json.dumps(PHASE10_OUTPUT, indent=2))
print(f'\n✅ Phase 10 output saved → {phase10_path}')


/tmp/ipykernel_11459/523027481.py:9: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Question: Which candidate is best suited for a machine learning role?


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



Answer: The first candidate, "ML engineer skilled in PyTorch, transformers, and data pipelines", is best suited for a machine learning role. This candidate's skills are directly related to machine learning, with expertise in PyTorch (a popular ML framework), transformers (a key architecture in natural language processing), and data pipelines (essential for data preparation and management in ML projects). 

In contrast, the second candidate has skills in Python development with FastAPI and AWS, which are more focused on web development and cloud computing, rather than machine learning specifically. While they may have some relevant skills, such as Python proficiency, they are not as directly suited for a machine learning role as the first candidate.

✅ Phase 10 output saved → /content/automated_interview_system/outputs/phase10_rag.json


## 📊 Phase 11 — Consolidated Summary Report

In [44]:
summary = {
    'project': 'Automated Candidate Interview & Evaluation System',
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'stack': {
        'llm_provider': 'Groq',
        'models': ['llama-3.1-8b-instant', 'llama-3.3-70b-versatile'],
        'vector_db': 'ChromaDB (free, local)',
        'embeddings': 'HuggingFace all-MiniLM-L6-v2 (free)',
        'langchain_version': '>=0.1.20 (1.2+ style imports)',
        'memory': 'Mem0 + ChromaDB',
    },
    'phases_completed': [
        'Phase 3: Pydantic Models',
        'Phase 4: Query Validation',
        'Phase 5: Logging',
        'Phase 6: ChromaDB Vector Store',
        'Phase 7: LangChain Resume Analyzer',
        'Phase 8: Multi-Agent Interview',
        'Phase 9: Mem0 Observability',
        'Phase 10: RAG over Candidate Profiles',
    ],
    'files_created': [str(f.relative_to(BASE_DIR)) for f in sorted(BASE_DIR.rglob('*.py'))],
    'interview_result': interview_result['final_report'],
}

summary_path = BASE_DIR / 'outputs' / 'SUMMARY_REPORT.json'
summary_path.write_text(json.dumps(summary, indent=2, default=str))

print('=' * 60)
print('✅ ALL PHASES COMPLETED SUCCESSFULLY')
print('=' * 60)
print(f'Project root : {BASE_DIR}')
print(f'Outputs dir  : {BASE_DIR / "outputs"}')
print(f'DB dir       : {BASE_DIR / "db"}')
print(f'Logs dir     : {BASE_DIR / "logs"}')
print()
for p in sorted((BASE_DIR / 'outputs').glob('*.json')):
    print(f'  📄 {p.name}')


✅ ALL PHASES COMPLETED SUCCESSFULLY
Project root : /content/automated_interview_system
Outputs dir  : /content/automated_interview_system/outputs
DB dir       : /content/automated_interview_system/db
Logs dir     : /content/automated_interview_system/logs

  📄 SUMMARY_REPORT.json
  📄 phase10_rag.json
  📄 phase3_pydantic.json
  📄 phase4_query.json
  📄 phase5_logging.json
  📄 phase6_chromadb.json
  📄 phase7_langchain.json
  📄 phase8_interview.json
  📄 phase9_mem0.json


/tmp/ipykernel_11459/2167461395.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': datetime.utcnow().isoformat() + 'Z',


## 📦 Phase 12 — Zip Everything & Download
Creates a single zip with all `.py` files, outputs, logs, and ChromaDB.


In [45]:
import zipfile, shutil
from pathlib import Path

ZIP_PATH = Path('/content/automated_interview_system_output.zip')

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in sorted(BASE_DIR.rglob('*')):
        # Skip __pycache__ and .pyc files
        if '__pycache__' in str(file_path) or file_path.suffix == '.pyc':
            continue
        if file_path.is_file():
            arcname = 'automated_interview_system/' + str(file_path.relative_to(BASE_DIR))
            zf.write(file_path, arcname)

zip_size = ZIP_PATH.stat().st_size / 1024
print(f'✅ Zip created: {ZIP_PATH}')
print(f'   Size: {zip_size:.1f} KB')
print()
print('Files in zip:')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    for name in sorted(zf.namelist()):
        info = zf.getinfo(name)
        print(f'  {name} ({info.file_size} bytes)')


✅ Zip created: /content/automated_interview_system_output.zip
   Size: 30.5 KB

Files in zip:
  automated_interview_system/agent_core.py (2185 bytes)
  automated_interview_system/db/c4ab8614-bd84-4508-8ce2-1d78dd462842/data_level0.bin (167600 bytes)
  automated_interview_system/db/c4ab8614-bd84-4508-8ce2-1d78dd462842/header.bin (100 bytes)
  automated_interview_system/db/c4ab8614-bd84-4508-8ce2-1d78dd462842/length.bin (400 bytes)
  automated_interview_system/db/c4ab8614-bd84-4508-8ce2-1d78dd462842/link_lists.bin (0 bytes)
  automated_interview_system/db/chroma.sqlite3 (196608 bytes)
  automated_interview_system/logs/utility_logging_example.log (661 bytes)
  automated_interview_system/main.py (864 bytes)
  automated_interview_system/outputs/SUMMARY_REPORT.json (1442 bytes)
  automated_interview_system/outputs/phase10_rag.json (918 bytes)
  automated_interview_system/outputs/phase3_pydantic.json (262 bytes)
  automated_interview_system/outputs/phase4_query.json (836 bytes)
  automated_in

In [46]:
# Download the zip (Colab only)
try:
    from google.colab import files
    files.download(str(ZIP_PATH))
    print('⬇️  Download started!')
except Exception:
    print(f'ℹ️  Zip is at: {ZIP_PATH}')
    print('In Colab: Files panel (📁) → right-click → Download')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⬇️  Download started!
